In [ ]:
!pip install pandas scikit-learn

In [ ]:
import urllib.request
import pandas as pd

urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/ops4life/mlops-get-started/main/datasets/HR-Employee-Attrition.csv",
    "HR-Employee-Attrition.csv"
)

# Run prior pipeline steps to produce featured.csv
df_raw = pd.read_csv("HR-Employee-Attrition.csv")

df_fe = df_raw.copy()
df_fe['Attrition'] = df_fe['Attrition'].map({'Yes': 1, 'No': 0}).astype('int')
df_fe['OverTime'] = df_fe['OverTime'].map({'Yes': 1, 'No': 0}).astype('Int64')
df_fe['Gender'] = df_fe['Gender'].map({'Male': 1, 'Female': 0}).astype('Int64')
df_fe['BusinessTravel'] = df_fe['BusinessTravel'].map({'Non-Travel': 0, 'Travel_Rarely': 1, 'Travel_Frequently': 2}).astype('Int64')
df_fe['MaritalStatus'] = df_fe['MaritalStatus'].map({'Single': 0, 'Married': 1, 'Divorced': 2}).astype('Int64')
df_fe['OverallSatisfaction'] = (
    (df_fe['JobSatisfaction'] + df_fe['EnvironmentSatisfaction'] + df_fe['RelationshipSatisfaction'] + df_fe['WorkLifeBalance']) / 4
).round().astype('Int64')
df_fe = df_fe.drop(columns=['JobSatisfaction', 'EnvironmentSatisfaction', 'RelationshipSatisfaction', 'WorkLifeBalance'])
df_fe['AnnualIncome'] = pd.cut(df_fe['MonthlyIncome'] * 12, bins=[0, 30000, 60000, 90000, 120000, float('inf')], labels=[0, 1, 2, 3, 4], include_lowest=True).astype('Int64')
df_fe = df_fe.drop(columns=['MonthlyIncome'])
df_fe['AgeGroup'] = pd.cut(df_fe['Age'], bins=[17, 25, 35, 45, 60, 65], labels=[1, 2, 3, 4, 5]).astype('Int64')
df_fe = df_fe.drop(columns=['Age'])
df_fe['RoleStagnationRatio'] = (df_fe['YearsInCurrentRole'] / (df_fe['YearsAtCompany'] + 1)).round(3)
df_fe['TenureGap'] = df_fe['TotalWorkingYears'] - df_fe['YearsAtCompany']
df_fe['EarlyCompanyTenureRisk'] = (df_fe['YearsAtCompany'] <= 2).astype('Int64')
df_fe['LongTenureLowRoleRisk'] = ((df_fe['TotalWorkingYears'] > 5) & (df_fe['JobLevel'] <= 2)).astype('Int64')
drop_cols = [c for c in ['EmployeeNumber', 'EmployeeCount', 'StandardHours', 'Over18', 'JobRole', 'EducationField', 'Department'] if c in df_fe.columns]
df_fe = df_fe.drop(columns=drop_cols)

from sklearn.model_selection import train_test_split
X = df_fe.drop(columns=['Attrition'])
y = df_fe['Attrition']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train = X_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
train_df = pd.concat([X_train, y_train], axis=1)
train_df.to_csv("train.csv", index=False)
print("Setup complete: train.csv ready.")

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split


def preprocess_data(df):
    df_pp = df.copy()

    # Separate features and target
    X = df_pp.drop(columns=['Attrition'])
    y = df_pp['Attrition']

    # Split the data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    # Reset indices
    X_train = X_train.reset_index(drop=True)
    X_test = X_test.reset_index(drop=True)
    y_train = y_train.reset_index(drop=True)
    y_test = y_test.reset_index(drop=True)

    train_df = pd.concat([X_train, y_train], axis=1)
    test_df = pd.concat([X_test, y_test], axis=1)

    # Save preprocessed data
    train_df.to_csv("train.csv", index=False)
    test_df.to_csv("test.csv", index=False)

    print(train_df.head())
    print("Preprocessing completed and data saved.")
    return train_df, test_df


df = pd.read_csv("featured.csv")
preprocess_data(df)